# 08 · Pendekatan sederhana gabungan (eksploratif)
Embedding harian → PCA (fit di hari Train) → [opsional] lag 1-3 + rata-rata 3/5 hari tiap komponen → langsung ke model.
Faktor yang dibandingkan: **bobot** (plain vs berbobot skor kamus geopolitik `g`) × **temporal** (tanpa vs dengan). Sumber: embedding yang ada di cache.
Model: MLP (hyperparameter tetap, 5 seed) + Ridge/ElasticNet/SVR/GradientBoosting (tuned, protokol 03). Hanya butuh `data/export/`.

In [1]:
import sys; sys.path.insert(0, "../src")
import re, time, joblib
import numpy as np, pandas as pd
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor
import t3
from t3 import CFG, OUT, PRICE, SEED
from eval_utils import *

K = 10                                           # jumlah komponen PCA (tetap)
prices, articles, sent = t3.load_export(); days = sent.index
date = articles.aligned_trading_date.values
train_end = prices.loc[prices.split == "train", "date"].max()
SOURCES = [t for t in [*CFG["emb_tags"], "deberta_base"] if (t3.EXPORT / f"embeddings_{t}.npy").exists()]
print("sumber:", SOURCES)

sumber: ['bge_small', 'bge_base', 'deberta_base']


## Skor geopolitik g per artikel (kamus `src/geo_terms.txt`, biner) → bobot harian

In [2]:
terms = [t.strip().lower() for t in (t3.ROOT / "src/geo_terms.txt").read_text().splitlines() if t.strip() and not t.startswith("#")]
pat = re.compile(r"\b(?:" + "|".join(re.escape(t) for t in sorted(set(terms), key=len, reverse=True)) + r")(?:s|es|ed|ing)?\b")
text = (articles.title + " " + articles.full_text).str.lower()
g = text.map(lambda s: len(pat.findall(s))) / text.map(lambda s: len(s.split()))
wsum, cnt = g.groupby(date).transform("sum"), g.groupby(date).transform("size")
wn = pd.Series(np.where(wsum > 0, g / wsum.where(wsum > 0, 1), 1 / cnt), index=articles.index)   # bobot per hari, fallback seragam
print(f"g mean {g.mean():.4f} | artikel tanpa kata kamus {(g == 0).mean():.1%}")

g mean 0.0204 | artikel tanpa kata kamus 0.3%


## Fitur: PCA harian (+ temporal) per sumber × bobot

In [3]:
rows = lambda df: df.reindex(prices.date).set_axis(prices.index)
frames, SETS = [], {"A": PRICE}
for src in SOURCES:
    E = pd.DataFrame(t3.load_emb(src, articles)[0])
    for w in ("plain", "geo"):
        M = (E.mul(wn, axis=0).groupby(date).sum() if w == "geo" else E.groupby(date).mean()).reindex(days).fillna(0.0)
        has = (M.abs().sum(1) > 0).to_numpy()
        pca = PCA(K, random_state=SEED).fit(M[(days <= train_end) & has])              # PCA hanya dari hari Train
        Z = pd.DataFrame(pca.transform(M), index=days, columns=[f"{src}_{w}_pc{i}" for i in range(K)])
        Z.loc[~has] = 0.0                                                              # hari tanpa berita = netral
        T = pd.concat([Z.shift(l).add_suffix(f"_lag{l}") for l in (1, 2, 3)] + [Z.rolling(r, min_periods=1).mean().add_suffix(f"_roll{r}") for r in (3, 5)], axis=1).fillna(0.0)
        frames += [rows(Z), rows(T)]
        SETS[f"{src}|{w}|static"] = PRICE + list(Z.columns)
        SETS[f"{src}|{w}|temporal"] = PRICE + list(Z.columns) + list(T.columns)
X2 = pd.concat([prices[["date", "split", "ret", "target", *PRICE]], *frames], axis=1)
tr, va, te = (X2[X2.split == s] for s in ("train", "val", "test"))
print(len(SETS), "set fitur | kolom terbanyak:", max(map(len, SETS.values())))

13 set fitur | kolom terbanyak: 67


## Latih: MLP (tetap) + 4 model tuned (resumable)

In [4]:
mlp = lambda seed: MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1e-2, max_iter=300, random_state=seed)
CACHE = OUT / "tuned_simple.joblib"
TUNED = joblib.load(CACHE) if CACHE.exists() else {}
fitted = {}
t0 = time.time()
for s, cols in SETS.items():
    fitted[("MLP", s)] = [t3.make_pipe(cols, mlp(sd)).fit(tr, tr.target) for sd in range(SEED, SEED + 5)]
    for m in t3.MODELS:
        if (m, s) not in TUNED:
            TUNED[(m, s)] = t3.tune(m, cols, tr, tr.target); joblib.dump(TUNED, CACHE)
        fitted[(m, s)] = TUNED[(m, s)]["models"]
    print(s, f"{time.time() - t0:.0f}s", flush=True)

A 18s
bge_small|plain|static 29s
bge_small|plain|temporal 51s
bge_small|geo|static 67s
bge_small|geo|temporal 89s
bge_base|plain|static 103s
bge_base|plain|temporal 125s
bge_base|geo|static 140s
bge_base|geo|temporal 162s
deberta_base|plain|static 177s
deberta_base|plain|temporal 201s
deberta_base|geo|static 216s
deberta_base|geo|temporal 240s


## Hasil (Val dan Test)

In [5]:
res = []
for (m, s), models in fitted.items():
    src, w, t = (s.split("|") + ["-", "-"])[:3]
    for name, D in (("Val", va), ("Test", te)):
        ev = pd.DataFrame([evaluate(D.target, x.predict(D)) for x in models])
        res.append({"model": m, "set": s, "src": src, "w": w, "t": t, "split": name, **ev.mean()[["RMSE", "DirAcc", "PredStd"]],
                    "collapse": is_collapsed(t3.predict(models, D), D.target)})
res = pd.DataFrame(res); res.to_csv(OUT / "simple_combined_results.csv", index=False)
for name, D in (("Val", va), ("Test", te)):
    print(name, "| Prediksi 0 RMSE:", round(evaluate(D.target, np.zeros(len(D)))["RMSE"], 6), "| DirAcc mayoritas:", round(majority_baselines(D.target)["Mayoritas"], 3))
res.pivot_table(index=["model", "set"], columns="split", values=["RMSE", "DirAcc"], sort=False).round(6)

Val | Prediksi 0 RMSE: 0.003382 | DirAcc mayoritas: 0.534
Test | Prediksi 0 RMSE: 0.002908 | DirAcc mayoritas: 0.591


RMSE              DirAcc  \
split                                              Val      Test       Val   
model            set                                                         
MLP              A                            0.003624  0.003230  0.513636   
                 bge_small|plain|static       0.004231  0.003539  0.464773   
                 bge_small|plain|temporal     0.004924  0.004147  0.478409   
                 bge_small|geo|static         0.004276  0.003609  0.470455   
                 bge_small|geo|temporal       0.004930  0.004160  0.489773   
...                                                ...       ...       ...   
GradientBoosting bge_base|geo|temporal        0.003400  0.002959  0.539773   
                 deberta_base|plain|static    0.003365  0.002945  0.517045   
                 deberta_base|plain|temporal  0.003369  0.002962  0.477273   
                 deberta_base|geo|static      0.003359  0.002926  0.556818   
                 deberta_base|geo|temporal    0.003394  0.002949  0.534091   

                                                        
split                                             Test  
model            set                                    
MLP              A                            0.486364  
                 bge_small|plain|static       0.526136  
                 bge_small|plain|temporal     0.536364  
                 bge_small|geo|static         0.529545  
                 bge_small|geo|temporal       0.500000  
...                                                ...  
GradientBoosting bge_base|geo|temporal        0.562500  
                 deberta_base|plain|static    0.568182  
                 deberta_base|plain|temporal  0.539773  
                 deberta_base|geo|static      0.613636  
                 deberta_base|geo|temporal    0.568182  

[65 rows x 4 columns]

## Apakah kamus (bobot) dan temporal berpengaruh?
Selisih RMSE rata-rata per model (negatif = faktor itu membantu) dan porsi sel yang membaik. Bandingkan dengan RMSE_std MLP (~1e-4) sebagai kisaran noise.

In [6]:
def effect(factor, a, b):
    d = res[res.src != "A"].pivot_table(index=["model", "src", "w" if factor == "t" else "t", "split"], columns=factor, values="RMSE")
    d = (d[a] - d[b]).rename("dRMSE").reset_index()
    return d.groupby(["model", "split"]).dRMSE.agg(mean="mean", share_better=lambda x: (x < 0).mean()).unstack("split").round(6)
print("Efek bobot kamus (geo − plain):"); display(effect("w", "geo", "plain"))
print("Efek temporal (temporal − static):"); display(effect("t", "temporal", "static"))

Efek bobot kamus (geo − plain):


mean           share_better          
split                 Test       Val         Test       Val
model                                                      
ElasticNet        0.000000  0.000000     0.000000  0.333333
GradientBoosting  0.000013  0.000016     0.333333  0.333333
MLP               0.000140 -0.000016     0.166667  0.333333
Ridge            -0.000000  0.000000     0.500000  0.333333
SVR               0.000002  0.000001     0.166667  0.000000

Efek temporal (temporal − static):


mean           share_better          
split                 Test       Val         Test       Val
model                                                      
ElasticNet        0.000000  0.000000     0.000000  0.000000
GradientBoosting -0.000003  0.000033     0.500000  0.000000
MLP               0.000644  0.000797     0.000000  0.000000
Ridge             0.000000  0.000001     0.000000  0.333333
SVR               0.000000  0.000003     0.333333  0.000000

In [7]:
top = res[(res.split == "Val") & ~res.collapse].sort_values("RMSE").head(8)[["model", "set"]]
t_ = res[res.split == "Test"].set_index(["model", "set"]).RMSE
top.assign(Val_RMSE=res[res.split == "Val"].set_index(["model", "set"]).RMSE.reindex(pd.MultiIndex.from_frame(top)).values,
           Test_RMSE=t_.reindex(pd.MultiIndex.from_frame(top)).values).round(6)

,model,set,Val_RMSE,Test_RMSE
118,GradientBoosting,deberta_base|geo|static,0.003359,0.002926
38,GradientBoosting,bge_small|geo|static,0.003361,0.002950
8,GradientBoosting,A,0.003362,0.002956
98,GradientBoosting,deberta_base|plain|static,0.003365,0.002945
58,GradientBoosting,bge_base|plain|static,0.003367,0.002930
108,GradientBoosting,deberta_base|plain|temporal,0.003369,0.002962
78,GradientBoosting,bge_base|geo|static,0.003373,0.002982
18,GradientBoosting,bge_small|plain|static,0.003379,0.002940
